# South Korea Power Grid V2 quickstart

This notebook shows a memory-conscious way to start with the unified Korea Power Exchange (KPX) 5-minute dataset. V2 provides the full normalized history in one Parquet file and one equivalent CSV file. For analysis, use the Parquet file: it is typed, compressed, and much smaller than the compatibility CSV.

In [ ]:
from datetime import datetime
from pathlib import Path
import json

import matplotlib.pyplot as plt
import pandas as pd
import pyarrow.dataset as ds
import pyarrow.parquet as pq

DATA_DIR = Path('/kaggle/input/south-korea-power-grid-5-minute')
PARQUET = DATA_DIR / 'south_korea_power_grid_5min.parquet'
MANIFEST = DATA_DIR / 'release_manifest.json'

assert PARQUET.exists(), f'Missing V2 Parquet: {PARQUET}'
assert MANIFEST.exists(), f'Missing release manifest: {MANIFEST}'
print(PARQUET)

## Schema and release size

The long-form schema has four columns. `source` determines what `value_mw` means: demand forecast, economic-dispatch BASEPOINT, or state-estimated generation. Generator IDs remain source-native; the dataset does not invent a crosswalk between dispatch and state estimation.

In [ ]:
pf = pq.ParquetFile(PARQUET)
print(f'Rows: {pf.metadata.num_rows:,}')
print(f'Row groups: {pf.metadata.num_row_groups:,}')
print(pf.schema_arrow)

## Verified source row counts

The release manifest contains the source-level counts produced by the local QA pipeline, so we can inspect them without scanning a billion rows.

In [ ]:
manifest = json.loads(MANIFEST.read_text(encoding='utf-8'))
source_counts = (
    pd.Series(manifest['source_rows'], name='rows')
    .rename_axis('source')
    .to_frame()
)
source_counts['share_pct'] = 100 * source_counts['rows'] / source_counts['rows'].sum()
source_counts

## Read one day of demand

Use Arrow filters so only the needed columns and time window are materialized. The full CSV is intentionally not loaded.

In [ ]:
grid = ds.dataset(PARQUET, format='parquet')
start = datetime(2026, 7, 1)
end = datetime(2026, 7, 2)
demand_filter = (
    (ds.field('source') == 'demand')
    & (ds.field('timestamp') >= start)
    & (ds.field('timestamp') < end)
)
demand = grid.to_table(
    columns=['timestamp', 'value_mw'],
    filter=demand_filter,
).to_pandas()
demand.head()

In [ ]:
ax = demand.plot(
    x='timestamp',
    y='value_mw',
    figsize=(12, 4),
    legend=False,
    title='KPX demand forecast — 2026-07-01',
)
ax.set_ylabel('MW')
ax.set_xlabel('timestamp')
plt.tight_layout()

## Inspect generator-level sources

Dispatch and state-estimation rows use the same four-column shape, but `value_mw` has source-specific semantics. These small filtered samples avoid loading either source in full.

In [ ]:
columns = ['timestamp', 'source', 'generator_id', 'value_mw']
dispatch_sample = grid.head(5, columns=columns, filter=ds.field('source') == 'dispatch').to_pandas()
state_sample = grid.head(5, columns=columns, filter=ds.field('source') == 'state_estimation').to_pandas()
print('Dispatch sample')
display(dispatch_sample)
print('State-estimation sample')
display(state_sample)

## Important data-quality notes

- Timestamps are timezone-naive because verified timezone metadata is not present in the source files.
- Missing five-minute timestamps remain missing; they are not silently imputed.
- Eight official source-month attachments were unavailable and are listed in `missing_source_months.csv`.
- Exact duplicates were removed under the documented candidate-key rule.
- The ambiguous state-estimation snapshot at `2016-06-03 17:20` was excluded rather than choosing arbitrarily between conflicting full-generator snapshots; see `normalization_exceptions.json`.
- Generator IDs are source-native. Do not assume a dispatch/state-estimation crosswalk unless you independently validate one.

For provenance, reuse notes, and integrity hashes, see `README.md`, `SOURCE_LICENSE.md`, `DATA_DICTIONARY.md`, and `release_manifest.json` in the dataset.